# HoloSyn: Multimodal P2P Co‑Regulation — Open‑Source Hugging Face Teacher Distillation (Colab)

Runs locally using **open-source Hugging Face models** (no API keys).

Pipeline:
1. Extract audio/video/image/text/haptics from `Archive.zip`
2. Run **teacher inference** with open-source models
3. Map teacher outputs → targets: `valence, arousal, calm, trust` (0..1)
4. Train a compact **student head** and export **TorchScript**
5. Optional runtime: **Cirq/qsimcirq** synchrony + **Brian2** entrainment


In [ ]:
#@title 0) Install dependencies
!pip -q install -U numpy pandas tqdm pillow opencv-python soundfile librosa
!pip -q install -U torch torchvision torchaudio
!pip -q install -U transformers accelerate sentencepiece safetensors
!pip -q install -U cirq qsimcirq brian2

import os, json, zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import torch
print("✅ Installed.")

In [ ]:
#@title 1) Locate and extract Archive.zip
def pick_existing(*cands):
    for c in cands:
        if c and os.path.exists(c):
            return c
    return None

ARCHIVE_ZIP = pick_existing("/mnt/data/Archive.zip", "/content/Archive.zip")
assert ARCHIVE_ZIP, "❌ Archive.zip not found. Upload it to Colab Files or place in /mnt/data."

EXTRACT_DIR = "/content/archive_extracted"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

print("ARCHIVE_ZIP:", ARCHIVE_ZIP)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("Top-level:")
for p in sorted(Path(EXTRACT_DIR).iterdir())[:60]:
    print(" -", p.name, "(dir)" if p.is_dir() else "(file)")

In [ ]:
#@title 2) Discover files by modality
AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file(): out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

all_files = walk_files(EXTRACT_DIR)
buckets={}
for f in all_files:
    buckets.setdefault(bucket(f), []).append(f)

for k in ["audio","video","image","text","haptics","other"]:
    print(f"{k:8s}", len(buckets.get(k, [])))
    for s in buckets.get(k, [])[:3]:
        print("   •", s.replace(EXTRACT_DIR + "/", ""))

In [ ]:
#@title 3) Local features (privacy-preserving)
import librosa, cv2

def rel(p): return p.replace(EXTRACT_DIR + "/", "")
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try: return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except: return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {"txt_len":float(len(s)),
            "txt_lines":float(s.count("\n")+1),
            "txt_exclaim":float(s.count("!")),
            "txt_question":float(s.count("?")),
            "txt_caps_ratio":float(sum(c.isupper() for c in s)/max(1,len(s)))}

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {"hapt_len":float(len(raw)),
            "hapt_has_intensity":1.0 if "intensity" in raw else 0.0,
            "hapt_has_freq":1.0 if ("hz" in raw or "freq" in raw) else 0.0}

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {"img_w":float(arr.shape[1]),"img_h":float(arr.shape[0]),
            "img_mean_r":float(mean[0]),"img_mean_g":float(mean[1]),"img_mean_b":float(mean[2]),
            "img_std_r":float(std[0]),"img_std_g":float(std[1]),"img_std_b":float(std[2])}

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

MAX_PER_MODALITY = 250
rows=[]
for p in tqdm(buckets.get("text", [])[:MAX_PER_MODALITY], desc="text"):
    rows.append({"path":p,"rel":rel(p),"modality":"text",**featurize_text(load_text(p))})
for p in tqdm(buckets.get("haptics", [])[:MAX_PER_MODALITY], desc="haptics"):
    rows.append({"path":p,"rel":rel(p),"modality":"haptics",**featurize_haptics(load_haptics_any(p))})
for p in tqdm(buckets.get("audio", [])[:MAX_PER_MODALITY], desc="audio"):
    rows.append({"path":p,"rel":rel(p),"modality":"audio",**audio_features(p)})
for p in tqdm(buckets.get("image", [])[:MAX_PER_MODALITY], desc="image"):
    try: rows.append({"path":p,"rel":rel(p),"modality":"image",**image_quick_stats(p)})
    except Exception as e: print("skip image", rel(p), e)
for p in tqdm(buckets.get("video", [])[:MAX_PER_MODALITY], desc="video"):
    try: rows.append({"path":p,"rel":rel(p),"modality":"video","vid_n_frames":float(len(sample_video_frames(p)))})
    except Exception as e: print("skip video", rel(p), e)

df = pd.DataFrame(rows).fillna(0.0)
print("✅ Base features:", df.shape)
df.head()

In [ ]:
#@title 4) Load open-source teachers (HF)
from transformers import pipeline, AutoProcessor, AutoModel
import torch

device = 0 if torch.cuda.is_available() else -1
print("device:", "cuda" if device==0 else "cpu")

EMO_MODEL = "j-hartmann/emotion-english-distilroberta-base"
SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
CLIP_MODEL = "openai/clip-vit-base-patch32"
W2V_MODEL  = "facebook/wav2vec2-base-960h"

emo_pipe = pipeline("text-classification", model=EMO_MODEL, top_k=None, device=device)
sent_pipe = pipeline("text-classification", model=SENT_MODEL, top_k=None, device=device)

clip_processor = AutoProcessor.from_pretrained(CLIP_MODEL)
clip_model = AutoModel.from_pretrained(CLIP_MODEL).to("cuda" if device==0 else "cpu").eval()

w2v_processor = AutoProcessor.from_pretrained(W2V_MODEL)
w2v_model = AutoModel.from_pretrained(W2V_MODEL).to("cuda" if device==0 else "cpu").eval()

print("✅ Teachers loaded")

In [ ]:
#@title 5) Teacher cache: targets + embeddings
def softmax_dict(items):
    d = {it["label"].lower(): float(it["score"]) for it in items}
    s = sum(d.values()) or 1.0
    return {k:v/s for k,v in d.items()}

def emotion_to_targets(emo_probs, sent_probs):
    valence = sent_probs.get("positive",0.33) + 0.5*sent_probs.get("neutral",0.33)
    valence = float(np.clip(valence,0,1))
    arousal = (
        1.0*emo_probs.get("anger",0) +
        1.0*emo_probs.get("fear",0) +
        0.8*emo_probs.get("surprise",0) +
        0.6*emo_probs.get("joy",0) +
        0.2*emo_probs.get("sadness",0) +
        0.3*emo_probs.get("disgust",0)
    )
    arousal = float(np.clip(arousal,0,1))
    calm = float(np.clip(1.0-arousal,0,1))
    trust = float(np.clip(valence*(1.0-0.8*emo_probs.get("fear",0)-0.6*emo_probs.get("anger",0)),0,1))
    return valence, arousal, calm, trust

@torch.no_grad()
def clip_image_embedding(pil_images):
    if not isinstance(pil_images, list): pil_images=[pil_images]
    inputs = clip_processor(images=pil_images, return_tensors="pt")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k:v.to(dev) for k,v in inputs.items()}
    out = clip_model.get_image_features(**inputs)
    out = out / out.norm(dim=-1, keepdim=True)
    return out.cpu().numpy()

@torch.no_grad()
def wav2vec_embedding(wav_path, sr=16000, max_seconds=10):
    y, _ = librosa.load(wav_path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return np.zeros((768,), dtype=np.float32)
    inputs = w2v_processor(y, sampling_rate=sr, return_tensors="pt")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k:v.to(dev) for k,v in inputs.items()}
    out = w2v_model(**inputs).last_hidden_state
    emb = out.mean(dim=1).squeeze(0)
    emb = emb / (emb.norm() + 1e-8)
    return emb.detach().cpu().numpy().astype(np.float32)

CACHE_PATH="/content/hf_teacher_cache.parquet"
teacher_df = pd.read_parquet(CACHE_PATH) if os.path.exists(CACHE_PATH) else pd.DataFrame()
done=set(teacher_df["rel"].tolist()) if len(teacher_df) else set()

records=[]
N_PER_MODALITY=180

# text/haptics targets
for modality in ["text","haptics"]:
    sub = df[df["modality"]==modality].head(N_PER_MODALITY)
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"teacher {modality}"):
        if row["rel"] in done: continue
        try:
            txt = load_text(row["path"]) if modality=="text" else ("HAPTICS_JSON:\n"+json.dumps(load_haptics_any(row["path"]))[:12000])
            emo = emo_pipe(txt[:3000])[0]
            sent = sent_pipe(txt[:3000])[0]
            emo_probs, sent_probs = softmax_dict(emo), softmax_dict(sent)
            v,a,c,t = emotion_to_targets(emo_probs, sent_probs)
            records.append({"rel":row["rel"],"modality":modality,"valence":v,"arousal":a,"calm":c,"trust":t})
        except Exception as e:
            print("skip", row["rel"], e)

# image/video clip embeddings
for modality in ["image","video"]:
    sub = df[df["modality"]==modality].head(N_PER_MODALITY)
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"teacher {modality}"):
        if row["rel"] in done: continue
        try:
            if modality=="image":
                ims=[Image.open(row["path"]).convert("RGB")]
            else:
                frames=sample_video_frames(row["path"])
                if not frames: continue
                ims=[Image.fromarray(f).convert("RGB") for f in frames]
            embs=clip_image_embedding(ims)
            emb=embs.mean(axis=0)
            emb=emb/(np.linalg.norm(emb)+1e-8)
            records.append({"rel":row["rel"],"modality":modality,
                            "valence":0.5,"arousal":0.5,"calm":0.5,"trust":0.5,
                            "clip_emb":emb.tolist()})
        except Exception as e:
            print("skip", modality, row["rel"], e)

# audio wav2vec embedding
sub = df[df["modality"]=="audio"].head(N_PER_MODALITY)
for _, row in tqdm(sub.iterrows(), total=len(sub), desc="teacher audio"):
    if row["rel"] in done: continue
    try:
        emb=wav2vec_embedding(row["path"])
        records.append({"rel":row["rel"],"modality":"audio",
                        "valence":0.5,"arousal":0.5,"calm":0.5,"trust":0.5,
                        "w2v_emb":emb.tolist()})
    except Exception as e:
        print("skip audio", row["rel"], e)

new_df=pd.DataFrame(records)
teacher_df = pd.concat([teacher_df, new_df], ignore_index=True) if len(teacher_df) else new_df
teacher_df.to_parquet(CACHE_PATH, index=False)
print("✅ Teacher cache:", teacher_df.shape, "saved to", CACHE_PATH)
teacher_df.head()

In [ ]:
#@title 6) Train student head + export TorchScript
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

merged = df.merge(pd.read_parquet(CACHE_PATH), on=["rel","modality"], how="inner")

# expand embeddings
def expand_vec(colname, prefix):
    if colname not in merged.columns: return
    vecs = merged[colname].dropna()
    if len(vecs)==0: return
    dim = len(vecs.iloc[0])
    arr = np.vstack([v if isinstance(v, list) else [0.0]*dim for v in merged[colname].fillna([0.0]*dim)])
    for i in range(dim):
        merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)
    merged.drop(columns=[colname], inplace=True)

expand_vec("clip_emb","clip_")
expand_vec("w2v_emb","w2v_")

TARGETS=["valence","arousal","calm","trust"]
ignore=set(["path","rel","modality"]+TARGETS)
numeric_cols=[c for c in merged.columns if c not in ignore and pd.api.types.is_numeric_dtype(merged[c])]

X = merged[numeric_cols].astype(np.float32).values
Y = merged[TARGETS].astype(np.float32).values

mu=X.mean(axis=0, keepdims=True)
sd=X.std(axis=0, keepdims=True)+1e-6
Xn=(X-mu)/sd

class TableDS(Dataset):
    def __init__(self, X, Y):
        self.X=torch.tensor(X, dtype=torch.float32)
        self.Y=torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self,i): return self.X[i], self.Y[i]

ds=TableDS(Xn,Y)
dl=DataLoader(ds,batch_size=64,shuffle=True)

student = nn.Sequential(
    nn.Linear(Xn.shape[1],256), nn.Tanh(),
    nn.Linear(256,128), nn.Tanh(),
    nn.Linear(128,4), nn.Sigmoid()
)

dev="cuda" if torch.cuda.is_available() else "cpu"
student.to(dev)
opt=torch.optim.Adam(student.parameters(), lr=1e-3)
loss_fn=nn.MSELoss()

student.train()
for epoch in range(12):
    total=0.0
    for xb,yb in dl:
        xb,yb=xb.to(dev), yb.to(dev)
        pred=student(xb)
        loss=loss_fn(pred,yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*xb.size(0)
    print(f"epoch {epoch:02d} mse {total/len(ds):.6f}")

student_cpu = student.to("cpu").eval()
example=torch.zeros(1, Xn.shape[1])
ts=torch.jit.trace(student_cpu, example)
student_ts_path="/content/student_distilled_heads_hf.torchscript.pt"
ts.save(student_ts_path)

norm_path="/content/student_norm_hf.json"
with open(norm_path,"w",encoding="utf-8") as f:
    json.dump({"numeric_cols":numeric_cols,"mu":mu.flatten().tolist(),"sd":sd.flatten().tolist()}, f)

print("✅ Saved:", student_ts_path)
print("✅ Norm :", norm_path)

In [ ]:
#@title 7) Optional: Quantum synchrony + Brian2 entrainment
import cirq, qsimcirq
from brian2 import *

class QuantumSynchrony:
    def __init__(self, n_qubits=4, depth=2):
        self.n_qubits=n_qubits; self.depth=depth
        self.qubits=cirq.LineQubit.range(n_qubits)
        self.sim=qsimcirq.QSimSimulator()
    def _reduce(self, e):
        x=np.tanh(np.asarray(e,dtype=np.float32))*np.pi
        x=x[:self.n_qubits]
        if x.size<self.n_qubits: x=np.pad(x,(0,self.n_qubits-x.size))
        return x
    def _circuit(self, x):
        c=cirq.Circuit()
        c.append([cirq.H(q) for q in self.qubits])
        for _ in range(self.depth):
            for i,q in enumerate(self.qubits):
                c.append(cirq.ry(x[i])(q)); c.append(cirq.rz(0.5*x[i])(q))
            for i in range(self.n_qubits-1):
                c.append(cirq.CZ(self.qubits[i], self.qubits[i+1]))
        return c
    def kernel(self, ea, eb):
        sa=self.sim.simulate(self._circuit(self._reduce(ea))).final_state_vector
        sb=self.sim.simulate(self._circuit(self._reduce(eb))).final_state_vector
        return float(np.abs(np.vdot(sa,sb))**2)

def plv_from_coupling(coupling=0.2, duration_ms=200):
    start_scope()
    eqs='''dtheta/dt = omega + k*sin(theta_other - theta) : 1
    theta_other : 1
    omega : 1
    k : 1'''
    G=NeuronGroup(2, eqs, method='euler')
    G.theta=[0.1,2.0]; G.omega=[2*np.pi*8,2*np.pi*8]; G.k=coupling
    @network_operation(dt=1*ms)
    def couple():
        G.theta_other[0]=G.theta[1]; G.theta_other[1]=G.theta[0]
    M=StateMonitor(G,'theta',record=True)
    net=Network(G,couple,M); net.run(duration_ms*ms)
    phase_diff=np.array(M.theta[0]-M.theta[1])
    return float(np.abs(np.mean(np.exp(1j*phase_diff))))

qs=QuantumSynchrony()
print("qsync demo:", qs.kernel([0.2,0.1,0.6,0.4],[0.2,0.1,0.6,0.4]))
print("plv demo:", plv_from_coupling(0.05), plv_from_coupling(0.35))